# PAI最終課題: 未使用の評価集合でのモデル比較 (Colab)
gemma4:12b-it-q4_K_M と qwen3.5:9b-q4_K_M を `eval/cases_heldout.yaml`(凍結済み・未使用)で測る。
**ランタイムは GPU(T4以上) にしてから上から順に実行。** 評価集合を見てプロンプトを直さないこと。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import subprocess
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'GPUランタイムではありません'

In [ ]:
import base64, os, zipfile, io
B64 = "..."
os.makedirs('/content/pai-final', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(B64))).extractall('/content/pai-final')
%cd /content/pai-final
!python -m unittest discover -s tests 2>&1 | tail -3

In [ ]:
# Ollama(公式インストーラ)を入れて、ローカルでサーバを起動する(ポートは評価スクリプトの既定 11435)
!apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
import subprocess, time, urllib.request, os
env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11435', OLLAMA_KEEP_ALIVE='2m', OLLAMA_MAX_LOADED_MODELS='1')
srv = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:11435/api/version', timeout=2).read().decode()); break
    except Exception: time.sleep(1)

In [ ]:
import os
os.environ['OLLAMA_HOST'] = '127.0.0.1:11435'
!ollama pull gemma4:12b-it-q4_K_M 2>&1 | tail -2
!ollama pull qwen3.5:9b-q4_K_M 2>&1 | tail -2
!ollama list

In [ ]:
import json, subprocess, urllib.request
req = urllib.request.Request('http://127.0.0.1:11435/api/generate', headers={"Content-Type": "application/json"},
    data=json.dumps({"model": "gemma4:12b-it-q4_K_M", "prompt": "hi", "stream": False, "options": {"num_predict": 8}, "keep_alive": "2m"}).encode())
json.loads(urllib.request.urlopen(req, timeout=900).read())
ps = json.loads(urllib.request.urlopen('http://127.0.0.1:11435/api/ps').read())['models'][0]
used = int(subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.split()[0])
print('ollama ps size/size_vram:', ps['size'], ps['size_vram'], '| GPU used MiB:', used)
assert used > 4000, 'OllamaがGPUを使っていない(CPU実行)。ここで止める'
print('GPU_OK')

In [ ]:
# 測定: 各モデル1回(3シード x 20件)。結果は /content/pai-final/eval/results/*.heldout-colab.json
!python3 eval/bench_models.py --model gemma4:12b-it-q4_K_M --cases eval/cases_heldout.yaml --out eval/results --tag heldout-colab --seeds 1,2,3 --temperature 0.3
!python3 eval/bench_models.py --model qwen3.5:9b-q4_K_M --cases eval/cases_heldout.yaml --out eval/results --tag heldout-colab --seeds 1,2,3 --temperature 0.3
print('COLAB_BENCH_DONE')

In [ ]:
import json, glob
for f in sorted(glob.glob('eval/results/*.heldout-colab.json')):
    d = json.load(open(f))
    print(f); print(json.dumps(d['aggregate'], ensure_ascii=False))
    for cid, cr in d['case_rate'].items():
        print(' ', cid, f"{cr['correct']}/{cr['of']}", cr['got'], cr['replans'])
print('SUMMARY_DONE')